Estimador de arriendos


In [1]:
import numpy as np
import pandas as pd

In [29]:

n_muestras = 500
barrios = ['Centro', 'Ensanche', 'Periferia', 'Zona Norte']
certificados = ['A', 'B', 'C', 'D', 'E', 'F', 'G']

df = pd.DataFrame({
    'metros_cuadrados': np.random.randint(40, 200, n_muestras),
    'habitaciones': np.random.randint(1, 5, n_muestras),
    'banos': np.random.randint(1, 3, n_muestras),
    'planta': np.random.randint(0, 10, n_muestras),
    'ascensor': np.random.randint(0, 2, n_muestras),
    'antiguedad_anios': np.random.randint(0, 60, n_muestras),
    'certificado_energetico': np.random.choice(certificados, n_muestras),
    'barrio': np.random.choice(barrios, n_muestras),
    'distancia_centro_km': np.round(np.random.uniform(0.5, 15, n_muestras), 1),
})

# Precio "de verdad" fabricado a propósito, para comprobar que el modelo lo aprende
ajuste_barrio = df['barrio'].map({'Centro': 60000, 'Ensanche': 30000, 'Periferia': 0, 'Zona Norte': 15000})
ajuste_certificado = df['certificado_energetico'].map(
    {'A': 15000, 'B': 10000, 'C': 5000, 'D': 0, 'E': -5000, 'F': -10000, 'G': -15000}
)

ruido = np.random.randn(n_muestras) * 8000

df['precio'] = (
    50000
    + 1800 * df['metros_cuadrados']
    + 5000 * df['habitaciones']
    + 3000 * df['banos']
    - 1200 * df['antiguedad_anios']
    + 10000 * df['ascensor']
    - 2000 * df['distancia_centro_km']
    + ajuste_barrio
    + ajuste_certificado
    + ruido
).astype('int')

df.head()

,metros_cuadrados,habitaciones,banos,planta,ascensor,antiguedad_anios,certificado_energetico,barrio,distancia_centro_km,precio
0,108,3,2,2,0,11,A,Centro,3.5,324074
1,144,4,2,2,1,0,E,Periferia,8.2,314900
2,140,2,2,3,1,3,F,Zona Norte,14.2,305596
3,155,4,1,2,1,11,F,Periferia,1.0,321558
4,47,2,2,7,0,5,B,Ensanche,1.0,189391


In [30]:
df_codificado=pd.get_dummies(df,columns=['certificado_energetico','barrio'],drop_first=True)

In [32]:
df_codificado

,metros_cuadrados,habitaciones,banos,planta,ascensor,antiguedad_anios,distancia_centro_km,precio,certificado_energetico_B,certificado_energetico_C,certificado_energetico_D,certificado_energetico_E,certificado_energetico_F,certificado_energetico_G,barrio_Ensanche,barrio_Periferia,barrio_Zona Norte
0,108,3,2,2,0,11,3.5,324074,False,False,False,False,False,False,False,False,False
1,144,4,2,2,1,0,8.2,314900,False,False,False,True,False,False,False,True,False
2,140,2,2,3,1,3,14.2,305596,False,False,False,False,True,False,False,False,True
3,155,4,1,2,1,11,1.0,321558,False,False,False,False,True,False,False,True,False
4,47,2,2,7,0,5,1.0,189391,True,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,78,2,1,4,1,26,8.2,182111,False,False,True,False,False,False,False,False,True
496,133,2,1,9,0,36,2.8,262217,False,False,False,False,False,True,False,False,True
497,151,3,2,9,0,12,4.3,384839,False,False,False,False,True,False,False,False,False
498,123,3,2,4,0,40,3.1,297710,False,True,False,False,False,False,False,False,False


In [55]:
from sklearn.linear_model import LinearRegression

columnas_entrada = [c for c in df_codificado.columns if c != 'precio']

X = df_codificado[columnas_entrada].values
y = df_codificado['precio'].values

lin_reg=LinearRegression()
lin_reg.fit(X,y)

for nombre, coeficiente in zip(columnas_entrada, lin_reg.coef_):
    print(f"{nombre}: {coeficiente:.0f} €")


piso_nuevo = pd.DataFrame([{
    'metros_cuadrados': 95,
    'habitaciones': 3,
    'banos': 2,
    'planta': 3,
    'ascensor': 1,
    'antiguedad_anios': 10,
    'distancia_centro_km': 1.8,
    'certificado_energetico': 'B',
    'barrio': 'Ensanche',
}])

piso_nuevo_codificado = pd.get_dummies(piso_nuevo, columns=['certificado_energetico', 'barrio'], drop_first=True)
piso_nuevo_codificado = piso_nuevo_codificado.reindex(columns=columnas_entrada, fill_value=0)

precio_estimado = lin_reg.predict(piso_nuevo_codificado.values)
print("Precio estimado:", int(precio_estimado[0]), "€")

metros_cuadrados: 1802 €
habitaciones: 5448 €
banos: 2543 €
planta: -194 €
ascensor: 10265 €
antiguedad_anios: -1213 €
distancia_centro_km: -2055 €
certificado_energetico_B: -4895 €
certificado_energetico_C: -9264 €
certificado_energetico_D: -15686 €
certificado_energetico_E: -20566 €
certificado_energetico_F: -25901 €
certificado_energetico_G: -29814 €
barrio_Ensanche: -30071 €
barrio_Periferia: -60151 €
barrio_Zona Norte: -46436 €
Precio estimado: 312681 €
